# 01 — Audit Data Quality

**Goal**: Sample SGFs from each source, display raw comments, assess quality,
compute statistics, and identify normalization requirements before building
the harvest/refine pipeline.

**Data isolation**: This notebook reads from `data/sources/` (local copies).
Run `python -m tools.yen_sei ingest` first to copy SGFs from external-sources/.

**Sources** (after ingest):
- OGS curated (`data/sources/ogs/`) — ~178 SGFs
- GoProblems (`data/sources/goproblems/`) — 51,822 SGFs
- GoGameGuru (`data/sources/gogameguru/`) — 421 SGFs

In [ ]:
import os
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

# Add project root to path so we can import tools.core
PROJECT_ROOT = Path(os.getcwd()).resolve()
# Walk up until we find the yen-go root
while not (PROJECT_ROOT / "tools" / "core").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

# yen-sei data directory (local copies of SGFs)
YENSEI_DATA = PROJECT_ROOT / "tools" / "yen-sei" / "data"
SOURCES_DIR = YENSEI_DATA / "sources"

if not SOURCES_DIR.exists() or not any(SOURCES_DIR.iterdir()):
    print("ERROR: data/sources/ is empty.")
    print("Run this first:  python -m tools.yen_sei ingest")
    print(f"Expected at: {SOURCES_DIR}")
else:
    print(f"Project root: {PROJECT_ROOT}")
    print(f"Data sources: {SOURCES_DIR}")

In [ ]:
# Quick regex-based comment extractor (avoids sgf_parser dependency issues in notebook)
# We use this for the audit; the actual pipeline will use tools.core.sgf_parser

def extract_comments_from_sgf(sgf_text: str) -> list[dict]:
    """Extract all C[] properties from raw SGF text.
    
    Returns list of {"text": str, "position": "root" | "move", "index": int}
    """
    comments = []
    # Match C[...] handling escaped brackets
    pattern = r'C\[((?:[^\\\]]|\\.)*)\]'
    for i, match in enumerate(re.finditer(pattern, sgf_text)):
        text = match.group(1)
        # Unescape SGF escapes
        text = text.replace('\\]', ']').replace('\\\\', '\\')
        position = "root" if i == 0 else "move"
        comments.append({"text": text, "position": position, "index": i})
    return comments


def classify_comment(text: str) -> str:
    """Classify a comment by teaching quality."""
    stripped = text.strip().lower()
    
    # Marker-only
    markers = {
        "correct", "correct.", "wrong", "wrong.", "right", "right.",
        "+", "good", "good move", "bad", "bad move", "also correct",
        "also correct.", "incorrect", "incorrect.",
    }
    if stripped in markers:
        return "marker"
    
    # Brief (under 40 chars, no sentence structure)
    if len(stripped) < 40:
        return "brief"
    
    # Teaching (40+ chars, likely has explanation)
    if len(stripped) >= 80:
        return "teaching"
    
    return "moderate"


# Test on a known example
test_sgf = '(;C[Black to play.];B[rs]C[Correct];B[ns]C[Also correct. Playing this move at A is usually better style.])'
for c in extract_comments_from_sgf(test_sgf):
    print(f"  [{c['position']}] ({classify_comment(c['text'])}) {c['text'][:80]}")

## 1. Collect all SGF file paths by source

In [ ]:
# Read from data/sources/ (local copies, NOT external-sources/)
sources = {
    "ogs": SOURCES_DIR / "ogs",
    "goproblems": SOURCES_DIR / "goproblems",
    "gogameguru": SOURCES_DIR / "gogameguru",
}

sgf_files: dict[str, list[Path]] = {}
for name, root in sources.items():
    if root.exists():
        files = sorted(root.rglob("*.sgf"))
        sgf_files[name] = files
        print(f"{name}: {len(files)} SGF files")
    else:
        sgf_files[name] = []
        print(f"{name}: NOT FOUND at {root} — run 'python -m tools.yen_sei ingest --sources {name}'")

print(f"\nTotal: {sum(len(v) for v in sgf_files.values())} SGF files")

## 2. Sample and display raw comments from each source

In [ ]:
import random

random.seed(42)
SAMPLE_SIZE = 10  # per source

for source_name, files in sgf_files.items():
    if not files:
        continue
    
    sample = random.sample(files, min(SAMPLE_SIZE, len(files)))
    
    print(f"\n{'='*80}")
    print(f"SOURCE: {source_name} (sampling {len(sample)} of {len(files)})")
    print(f"{'='*80}")
    
    for sgf_path in sample:
        try:
            content = sgf_path.read_text(encoding="utf-8", errors="replace")
        except Exception as e:
            print(f"  ERROR reading {sgf_path.name}: {e}")
            continue
        
        comments = extract_comments_from_sgf(content)
        if not comments:
            print(f"\n  {sgf_path.name}: NO COMMENTS")
            continue
        
        print(f"\n  {sgf_path.name} ({len(comments)} comments):")
        for c in comments:
            cls = classify_comment(c["text"])
            preview = c["text"][:120].replace("\n", " ")
            if len(c["text"]) > 120:
                preview += "..."
            print(f"    [{c['position']:5s}] ({cls:8s}) {preview}")

## 3. Full scan — comment quality statistics

In [ ]:
# Scan ALL files (may take 1-2 minutes for 51K GoProblems files)

stats: dict[str, dict] = {}

for source_name, files in sgf_files.items():
    source_stats = {
        "total_files": len(files),
        "files_with_comments": 0,
        "files_with_teaching": 0,  # at least one 'teaching' or 'moderate' comment
        "comment_classes": Counter(),
        "comment_lengths": [],
        "teaching_comment_lengths": [],
        "format_issues": Counter(),  # HTML, weird encoding, etc.
    }
    
    for sgf_path in files:
        try:
            content = sgf_path.read_text(encoding="utf-8", errors="replace")
        except Exception:
            source_stats["format_issues"]["read_error"] += 1
            continue
        
        comments = extract_comments_from_sgf(content)
        if not comments:
            continue
        
        source_stats["files_with_comments"] += 1
        has_teaching = False
        
        for c in comments:
            text = c["text"]
            cls = classify_comment(text)
            source_stats["comment_classes"][cls] += 1
            source_stats["comment_lengths"].append(len(text))
            
            if cls in ("teaching", "moderate"):
                has_teaching = True
                source_stats["teaching_comment_lengths"].append(len(text))
            
            # Detect format issues
            if "<" in text and ">" in text:
                source_stats["format_issues"]["html_tags"] += 1
            if "\\n" in text:
                source_stats["format_issues"]["escaped_newlines"] += 1
            if "\ufffd" in text:
                source_stats["format_issues"]["encoding_errors"] += 1
        
        if has_teaching:
            source_stats["files_with_teaching"] += 1
    
    stats[source_name] = source_stats
    print(f"Scanned {source_name}: {len(files)} files")

print("\nDone!")

In [ ]:
# Display summary table

print(f"{'Source':<15} {'Total':>8} {'w/ Comments':>12} {'w/ Teaching':>12} {'Teaching %':>10}")
print("-" * 60)

total_teaching = 0
for source_name, s in stats.items():
    pct = (s['files_with_teaching'] / s['total_files'] * 100) if s['total_files'] > 0 else 0
    print(f"{source_name:<15} {s['total_files']:>8} {s['files_with_comments']:>12} {s['files_with_teaching']:>12} {pct:>9.1f}%")
    total_teaching += s['files_with_teaching']

print("-" * 60)
print(f"{'TOTAL':<15} {sum(s['total_files'] for s in stats.values()):>8} {'':<12} {total_teaching:>12}")

In [ ]:
# Comment classification breakdown per source

for source_name, s in stats.items():
    print(f"\n{source_name} — comment classification:")
    total_comments = sum(s["comment_classes"].values())
    for cls, count in s["comment_classes"].most_common():
        pct = count / total_comments * 100 if total_comments > 0 else 0
        print(f"  {cls:<12} {count:>8} ({pct:5.1f}%)")
    
    if s["format_issues"]:
        print(f"  Format issues:")
        for issue, count in s["format_issues"].most_common():
            print(f"    {issue}: {count}")

## 4. Comment length distribution

In [ ]:
# Text-based histogram (no matplotlib dependency)

def text_histogram(values: list[int], bins: list[int], label: str) -> None:
    """Print a simple text histogram."""
    if not values:
        print(f"  {label}: No data")
        return
    
    print(f"\n  {label} (n={len(values)}, median={sorted(values)[len(values)//2]}, max={max(values)}):")
    
    for i in range(len(bins) - 1):
        lo, hi = bins[i], bins[i + 1]
        count = sum(1 for v in values if lo <= v < hi)
        bar = '#' * (count * 50 // len(values)) if values else ''
        print(f"    {lo:>6}-{hi:<6} {count:>7} {bar}")
    
    overflow = sum(1 for v in values if v >= bins[-1])
    if overflow:
        bar = '#' * (overflow * 50 // len(values))
        print(f"    {bins[-1]:>6}+       {overflow:>7} {bar}")


length_bins = [0, 10, 20, 40, 80, 150, 300, 500, 1000]

for source_name, s in stats.items():
    text_histogram(s["comment_lengths"], length_bins, f"{source_name} — all comments")
    text_histogram(s["teaching_comment_lengths"], length_bins, f"{source_name} — teaching only")

## 5. Sample teaching-quality comments (best examples)

In [ ]:
# Show the best teaching comments from each source

random.seed(42)
MAX_EXAMPLES = 5

for source_name, files in sgf_files.items():
    print(f"\n{'='*80}")
    print(f"BEST TEACHING COMMENTS: {source_name}")
    print(f"{'='*80}")
    
    teaching_examples = []
    
    # Scan files for teaching comments
    sample_pool = random.sample(files, min(2000, len(files)))
    for sgf_path in sample_pool:
        try:
            content = sgf_path.read_text(encoding="utf-8", errors="replace")
        except Exception:
            continue
        
        for c in extract_comments_from_sgf(content):
            if classify_comment(c["text"]) == "teaching":
                teaching_examples.append((sgf_path.name, c["text"]))
    
    if not teaching_examples:
        print("  No teaching-quality comments found in sample.")
        continue
    
    # Show longest/best examples
    teaching_examples.sort(key=lambda x: len(x[1]), reverse=True)
    for filename, text in teaching_examples[:MAX_EXAMPLES]:
        print(f"\n  [{filename}] ({len(text)} chars):")
        # Show first 500 chars with proper formatting
        preview = text[:500]
        if len(text) > 500:
            preview += "..."
        for line in preview.split("\n"):
            print(f"    {line}")

## 6. Format issues analysis

In [ ]:
# Sample comments with HTML tags, encoding issues, or other format problems

random.seed(42)

for source_name, files in sgf_files.items():
    issues_found = []
    sample_pool = random.sample(files, min(2000, len(files)))
    
    for sgf_path in sample_pool:
        try:
            content = sgf_path.read_text(encoding="utf-8", errors="replace")
        except Exception:
            continue
        
        for c in extract_comments_from_sgf(content):
            text = c["text"]
            issues = []
            if "<" in text and ">" in text:
                issues.append("HTML")
            if "\ufffd" in text:
                issues.append("ENCODING")
            if re.search(r'https?://', text):
                issues.append("URL")
            if re.search(r'\[\d+\]', text):
                issues.append("BRACKET_NUMS")
            
            if issues:
                issues_found.append((sgf_path.name, issues, text[:200]))
    
    print(f"\n{source_name}: {len(issues_found)} comments with format issues (in sample)")
    for filename, issues, preview in issues_found[:5]:
        print(f"  [{filename}] {', '.join(issues)}:")
        print(f"    {preview[:150]}")

## 7. Normalization recommendations

Based on the audit above, document what normalization rules the `refine` stage needs.
Run the cells above first, then fill in observations here.

In [ ]:
# Summary of findings — fill in after running the audit

print("""NORMALIZATION RULES TO IMPLEMENT IN refine STAGE:

1. STRIP HTML: Remove <b>, <i>, <br>, <p> tags → plain text
2. CLEAN URLS: Remove http/https links
3. FIX ENCODING: Handle \ufffd replacement chars, latin-1 fallback
4. NORMALIZE WHITESPACE: Collapse multiple newlines, strip leading/trailing
5. FILTER MARKERS: Exclude comments that are only 'Correct'/'Wrong'/'+'
6. MIN LENGTH: Require 80+ chars for at least one comment in the puzzle
7. COORDINATE FORMATS: Standardize GTP (D4) vs SGF (dg) references
8. DEDUP: Hash board position to remove duplicate puzzles across sources

These rules should be verified against the actual data above.
Adjust thresholds and patterns based on what you see in the samples.
""")